<a id="top"></a>

# Netflix Content Analytics: End-to-End Data Analysis with Python

**Auspify Technologies · Data Analysis Using Python Internship · Tasks 1–6**
**Prepared by:** Furqaan

---

## Executive Overview

This notebook completes **Task 1 : Data Cleaning & Preparation** tasks of the Auspify *Data Analysis Using Python* task list on a single Netflix catalog dataset. 

| # | Level | Task | What it delivers |
|---|-------|------|------------------|
| [1](#task1) | Easy | Data Cleaning & Preparation | Validated, analysis-ready dataset + cleaning + export |
| [2](#task2) | Easy | Content Type Analysis Dashboard | Movies vs TV Shows distribution, proportions, summary |
| [3](#task3) | Medium | Country-Wise Content Analysis | Country rankings, concentration, geographic map, insights |
| [4](#task4) | Medium | Trend Analysis by Release Year | Yearly trends, growth/decline phases, seasonality |
| [5](#task5) | Advanced | Content Rating & Genre Analysis | Audience segmentation, interactive visuals, statistical tests |
| [6](#task6) | Advanced | Business Insights Report | EDA, three dashboards, recommendations, executive summary |

### Notebook conventions
* **Data-quality decisions are explicit.** Each cleaning rule states what was found, what was done, and why.
* **Outputs** (cleaned CSV, all figures, summary files) are written to the `data/outputs/` folder next to this notebook.

### Requirements
`python >= 3.9` · `pandas` · `numpy` · `matplotlib` 

Install with: `conda install pandas numpy matplotlib`

### Data description

| Column | Description |
|--------|-------------|
| `show_id` | Unique identifier of the title |
| `type` | Content type: `Movie` or `TV Show` |
| `title` | Title of the content |
| `director` | Director name(s); missing values are stored as the text `Not Given` |
| `country` | Country of production; missing values are stored as the text `Not Given` |
| `date_added` | Date the title was added to the platform (`m/d/yyyy` text) |
| `release_year` | Original release year |
| `rating` | Maturity rating (e.g. `TV-MA`, `PG-13`) |
| `duration` | Runtime in minutes for movies, number of seasons for TV shows (text) |
| `listed_in` | Comma-separated genre / category tags |

<a id="task1"></a>

---

# Task 1 · Data Cleaning & Preparation  `Easy`

**Goal:** prepare and organise the Netflix dataset for business analytics and reporting.

| Step | Activity |
|------|----------|
| 1 | Import the dataset with Python and Pandas |
| 2 | Identify and handle missing values |
| 3 | Remove duplicate records and formatting inconsistencies |
| 4 | Standardise `Country`, `Rating` and `Type` |
| 5 | Export the cleaned dataset for analysis |



## 0. Environment Setup

In [19]:
from pathlib import Path
import numpy as np
import pandas as pd

## Step 1 · Import the dataset


In [2]:
# path to data directory
DATA_DIR = '../data/netflix_dataset.csv'
CLEANED_DATA_PATH = '../data/output/cleaned_data.csv' # store the path and use when data is cleaned

# check for the directory 
if Path.exists(DATA_DIR):
    print('the data directory exists, all good :)')
else:
    raise FileNotFoundError

# load the dataset using pandas 
df = pd.read_csv(filepath_or_buffer=DATA_DIR)

# viewing the 5 rows
df.head(5)

the data directory exists, all good :)


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [3]:
df.shape

(8790, 10)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8790 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8790 non-null   str  
 1   type          8790 non-null   str  
 2   title         8790 non-null   str  
 3   director      8790 non-null   str  
 4   country       8790 non-null   str  
 5   date_added    8790 non-null   str  
 6   release_year  8790 non-null   int64
 7   rating        8790 non-null   str  
 8   duration      8790 non-null   str  
 9   listed_in     8790 non-null   str  
dtypes: int64(1), str(9)
memory usage: 686.8 KB


**Finding:** the file reports zero nulls, yet `director` and `country` contain the placeholder text **`Not Given`**. Left untreated, `Not Given` would be counted as a real director and a real country in every later aggregation.

In [5]:
# find the total `Not given` labels from director adn country 
null_directors = df[df['director'] == 'Not Given']
null_country = df[df['country'] == 'Not Given']

# fill the missing to standard np.nan values 
df = df.replace(to_replace='Not Given',
               value=np.nan,
               inplace=True)

# count the total null values
df.isna().sum()

show_id            0
type               0
title              0
director        2588
country          287
date_added         0
release_year       0
rating             0
duration           0
listed_in          0
dtype: int64

## Step 2 : Handlin up the missing values 
**Approach** 

1. As we have the `director` column, we will set the director to *Unknown*.
2. For the `country` column, we will feel with `USA`

In [6]:
# filling the values
print('filling the director column with the Unknwon value !')
df.fillna({'director': 'Unknown'}, inplace=True)


print('Filling the country column with the most repeated country')
df.fillna(
    value={'country' : 'United States'},
    inplace=True
)
df.isna().sum()

filling the director column with the Unknwon value !
Filling the country column with the most repeated country


show_id         0
type            0
title           0
director        0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
dtype: int64

## Step 3 · Remove duplicates and formatting inconsistencies

### 3.1 Duplicate records

Two checks are used:

1. **Identifier duplicates** — is `show_id` unique?
2. **Business-key duplicates** — are there records identical in *every* column except the identifier? These are the same title ingested twice under different IDs.

In [7]:
df.columns


Index(['show_id', 'type', 'title', 'director', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in'],
      dtype='str')

In [8]:
# find the duplicate value 
duplicate_record = df.duplicated(subset=['title', 'director', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in'], keep='first')

# view the duplicate records 
duplicate_records = df[duplicate_record]
print(f' found {len(duplicate_records)} duplicate records from the {len(df)}')
duplicate_records

 found 3 duplicate records from the 8790


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
537,s5965,TV Show,9-Feb,Unknown,Pakistan,3/20/2019,2018,TV-14,1 Season,"International TV Shows, TV Dramas"
4260,s5966,Movie,22-Jul,Paul Greengrass,Norway,10/10/2018,2018,R,144 min,"Dramas, Thrillers"
4261,s5967,Movie,15-Aug,Swapnaneel Jayakar,India,3/29/2019,2019,TV-14,124 min,"Comedies, Dramas, Independent Movies"


In [9]:
# let's remove the duplicate records 
df = df.drop_duplicates(subset=['type', 'title', 'director', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in'], keep='first')
print(f'Removed all the {len(duplicate_records)} duplicate records ! dataset left : {len(df)}')
df.head()

Removed all the 3 duplicate records ! dataset left : 8787


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,9/25/2021,2020,PG-13,90 min,Documentaries
1,s3,TV Show,Ganglands,Julien Leclercq,France,9/24/2021,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act..."
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,9/24/2021,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries"
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,9/22/2021,2021,TV-PG,91 min,"Children & Family Movies, Comedies"
4,s8,Movie,Sankofa,Haile Gerima,United States,9/24/2021,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies"


In [10]:
df.info()

<class 'pandas.DataFrame'>
Index: 8787 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8787 non-null   str  
 1   type          8787 non-null   str  
 2   title         8787 non-null   str  
 3   director      8787 non-null   str  
 4   country       8787 non-null   str  
 5   date_added    8787 non-null   str  
 6   release_year  8787 non-null   int64
 7   rating        8787 non-null   str  
 8   duration      8787 non-null   str  
 9   listed_in     8787 non-null   str  
dtypes: int64(1), str(9)
memory usage: 755.1 KB


### let's process the below things:
1. convert the column **date_added** to the data type `datetime`
2. convert the column **release_year** to the integer datatype

In [11]:
# convert the date added column to the datetime formate 
df['date_added'] = pd.to_datetime(df.date_added, 
                                  format="%m/%d/%Y",
                                 errors='coerce')
print(f'Converted the date_added column to datatype date time : {df['date_added'].dtype}')


Converted the date_added column to datatype date time : datetime64[us]


In [12]:
# converting the release_year to int 
df['release_year'] = df['release_year'].astype(int)
df['release_year'].head()

0    2020
1    2021
2    2021
3    2021
4    1993
Name: release_year, dtype: int64

## Step 4 Standardized the rating, country and type 

In [13]:
# Standardized the country , type and the rating to the category data type  
df['country'] = df['country'].astype("category")
df['type'] = df['type'].astype("category")
df['rating'] = df['rating'].astype("category")

# fetch the information about the data 
df.info()

<class 'pandas.DataFrame'>
Index: 8787 entries, 0 to 8789
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   show_id       8787 non-null   str           
 1   type          8787 non-null   category      
 2   title         8787 non-null   str           
 3   director      8787 non-null   str           
 4   country       8787 non-null   category      
 5   date_added    8787 non-null   datetime64[us]
 6   release_year  8787 non-null   int64         
 7   rating        8787 non-null   category      
 8   duration      8787 non-null   str           
 9   listed_in     8787 non-null   str           
dtypes: category(3), datetime64[us](1), int64(1), str(5)
memory usage: 575.7 KB


In [14]:
# Extract duration value
df['duration_value'] = (
    df['duration']
    .str.extract(r'(\d+)')[0]
    .astype('Int64')
)

# Extract duration unit
df['duration_unit'] = (
    df['duration']
    .str.extract(r'(min|Season|Seasons)', flags=2)[0]
    .str.lower()
)

# drop the duration column
df = df.drop(labels=['duration'], axis=1)

# viewing the dataset
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,listed_in,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,Documentaries,90,min
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",1,season
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,"TV Dramas, TV Horror, TV Mysteries",1,season
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,"Children & Family Movies, Comedies",91,min
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,"Dramas, Independent Movies, International Movies",125,min


In [15]:
df.head()

,show_id,type,title,director,country,date_added,release_year,rating,listed_in,duration_value,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,Documentaries,90,min
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",1,season
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,"TV Dramas, TV Horror, TV Mysteries",1,season
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,"Children & Family Movies, Comedies",91,min
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,"Dramas, Independent Movies, International Movies",125,min


In [20]:
# saving the cleaned file
df.to_csv(path_or_buf=CLEANED_DATA_PATH, index=False)